# Phase 6B: Hybrid Retrieval Fusion and Learning-to-Rank

**Project**: Enhanced Legal Document Similarity and Case Retrieval Using Hybrid BERT, Legal Knowledge, and Ensemble Learning  
**Dataset**: `data/processed/IL-TUR-PCR/` (Indian Legal Prior Case Retrieval - PCR)  
**Objective**: Combine lexical, dense semantic, and legal knowledge representations using score fusion, rank fusion, and supervised learning-to-rank on the development candidate pool.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.evaluation.hybrid_evaluator import HybridEvaluator
from src.ranking.rrf_fusion import RRFFusion
from src.ranking.weighted_fusion import WeightedLinearFusion
from src.ranking.lightgbm_ranker import LightGBMRanker
from src.ranking.xgboost_ranker import XGBoostRanker
from src.ranking.logistic_stacking import LeakageSafeLogisticStacking

print('Environment and modules loaded successfully.')

## 1. Load Preprocessed Datasets (Train and Dev)

We inspect the training and development partitions generated in Phase 6A.

In [ ]:
train_path = PROJECT_ROOT / 'data/processed/IL-TUR-PCR/train/hybrid_features.parquet'
dev_path = PROJECT_ROOT / 'data/processed/IL-TUR-PCR/dev/hybrid_features.parquet'

train_df = pd.read_parquet(train_path).sort_values('query_id')
dev_df = pd.read_parquet(dev_path).sort_values('query_id')

print(f'Train Pairs: {len(train_df):,}, Queries: {train_df["query_id"].nunique()}, Positives: {train_df["label"].sum()}')
print(f'Dev Pairs: {len(dev_df):,}, Queries: {dev_df["query_id"].nunique()}, Positives: {dev_df["label"].sum()}')

## 2. Evaluate Individual Retrieval Baselines on Dev

We evaluate standalone BM25, TF-IDF, and InLegalBERT dense retrieval on the unconstrained development pool (1,022 candidates per query).

In [ ]:
m_bm25 = HybridEvaluator.evaluate_predictions(dev_df, score_col='bm25_score')
m_tfidf = HybridEvaluator.evaluate_predictions(dev_df, score_col='tfidf_score')
m_dense = HybridEvaluator.evaluate_predictions(dev_df, score_col='inlegalbert_sim')

baseline_df = pd.DataFrame([
    {'Model': 'BM25 Baseline', **m_bm25},
    {'Model': 'TF-IDF Baseline', **m_tfidf},
    {'Model': 'InLegalBERT Dense', **m_dense},
])
baseline_df[['Model', 'P@1', 'P@5', 'P@10', 'MRR', 'nDCG@5', 'nDCG@10']]

## 3. Model A: Reciprocal Rank Fusion (RRF)

RRF combines rank reciprocals: $RRF(d) = \sum_{m} \frac{1}{k + \text{rank}_m(d)}$ across BM25, TF-IDF, and InLegalBERT.

In [ ]:
rrf = RRFFusion(k=60, rank_cols=['bm25_rank', 'tfidf_rank', 'inlegalbert_rank'])
dev_df['score_rrf_60'] = rrf.compute_scores(dev_df)
m_rrf_60 = HybridEvaluator.evaluate_predictions(dev_df, score_col='score_rrf_60')

# Sweep k in {20, 60, 100}
k_sweep = RRFFusion.evaluate_k_grid(dev_df, k_values=[20, 60, 100], rank_cols=['bm25_rank', 'tfidf_rank', 'inlegalbert_rank'])
rrf_df = pd.DataFrame([{'k': k, **metrics} for k, metrics in k_sweep.items()])
rrf_df[['k', 'P@1', 'P@5', 'P@10', 'MRR', 'nDCG@5', 'nDCG@10']]

## 4. Model B: Weighted Linear Score Fusion

Query-level min-max normalization followed by vectorized grid search on the development set.

In [ ]:
weighted_fusion = WeightedLinearFusion(
    feature_cols=['bm25_score', 'tfidf_score', 'inlegalbert_sim', 'composite_legal_score'],
    use_query_min_max=True,
)
grid_res = weighted_fusion.fit_grid_search(dev_df, step=0.05, target_metric='nDCG@10')
print(f"Best Weights [BM25, TFIDF, InLegalBERT, LegalComposite]: {grid_res['best_weights']}")
pd.DataFrame([{'Model': 'Weighted Linear Fusion', **grid_res['full_metrics']}])[['Model', 'P@1', 'P@5', 'P@10', 'MRR', 'nDCG@5', 'nDCG@10']]

## 5. Model C & D: Tree-Based Learning-to-Rank (LightGBM & XGBoost)

Trained on query-grouped pairwise features with listwise/pairwise objectives.

In [ ]:
# Load pre-evaluated comparison table
comp_df = pd.read_csv(PROJECT_ROOT / 'results/tables/hybrid_model_comparison.csv')
comp_df[['Model', 'Type', 'P@1', 'P@5', 'P@10', 'MRR', 'nDCG@5', 'nDCG@10']]

## 6. Model E: Leakage-Safe Logistic Stacking Meta-Learner

5-Fold GroupKFold Out-of-Fold training partitioned by query ID.

In [ ]:
with open(PROJECT_ROOT / 'results/evaluation/hybrid_metrics.json') as f:
    metrics_json = json.load(f)

stack_meta = metrics_json['model_e_logistic_stacking']['meta_coefficients']
print('Learned Meta-Learner Coefficients:')
for k, v in stack_meta.items():
    print(f'  {k}: {v}')

## 7. Feature Ablation Study (7 Regimes)

We analyze the incremental value of Lexical, Dense, and Legal features.

In [ ]:
abl_df = pd.read_csv(PROJECT_ROOT / 'results/tables/hybrid_ablation_results.csv')
abl_df[['Regime', 'Num_Features', 'P@1', 'P@5', 'P@10', 'MRR', 'nDCG@5', 'nDCG@10']]

In [ ]:
# Visualization of Ablation Results
plt.figure(figsize=(10, 5))
sns.barplot(data=abl_df, x='Regime', y='MRR', palette='Blues_r')
plt.title('Ablation Study: Mean Reciprocal Rank (MRR) across Feature Regimes')
plt.xticks(rotation=30, ha='right')
plt.ylabel('MRR')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()